# 為研究工作流新增元件級評估

## 1. 引言

假設你已經有一個會使用工具的研究型代理，執行由三步組成的工作流：

1. 在網上搜尋資訊。  
2. 對其輸出進行反思。  
3. 釋出一份清晰的 HTML 報告。  

在本範例中，你將專注於評估該工作流中的**一個元件**：即*研究步驟*。  

不同於生成和潤色文章，這裡你將設計一個**元件級評估**，用於檢查研究步驟返回的來源質量。  

該評估會將代理檢索到的 URL 與一份**預定義的優選域名列表**進行對比（例如：`arxiv.org`、`nature.com`、`nasa.gov`）。  

這使你能夠用**客觀的、按示例的真實標準**來量化系統是否從可信來源獲取資訊。


### 1.1. 範例概覽

網頁搜尋有時會返回**質量較差**的結果，使得檢索到的資訊難以被信任。  
因此，在本範例中你將透過與**預定義的優選域名列表**對比來評估來源的可靠性。

在此次評估中，我們將聚焦主題*“黑洞科學的最新進展”*。  
我們的想法是驗證網頁搜尋工具是否返回了來自優選域名的來源，並量化優選結果與總結果的比例。

該評估將以一個函式實現，執行**客觀的、按示例的檢查**：

* 解析 Tavily 的輸出（我們的網頁搜尋工具）。  
* 識別哪些 URL 屬於**優選域名**列表。  
* 計算優選來源佔已檢索來源的比例。  
* 返回一個布林標誌（**透過/未透過**），以及可直接嵌入報告的 Markdown 格式摘要。  

<img src="M4-UGL-1.png" width="70%">  


### 1.2. 🎯 學習目標

你將學到：

* 編寫一個函式，檢查網頁搜尋 API 的結果是否來自**優選來源**。  
* 建立一個評估，用於驗證你的來源是否來自**優選域名**。  
* 將一個**元件級評估**加入網頁搜尋函式。  


## 2. 環境設定：匯入庫並載入環境

你首先匯入所需的庫並初始化執行環境。

In [ ]:
# =========================
# 匯入
# =========================

# --- 標準庫 
from datetime import datetime
import json
import re

# --- 第三方 ---
from aisuite import Client

# --- 本地 / 專案 ---
import research_tools
import utils

client = Client()

## 3. 研究步驟 – `find_references`

在完整的研究工作流中，原本的函式在同一步中同時完成了**網頁搜尋與草稿報告撰寫**。

這裡我們將網頁搜尋拆分為一個名為 `find_references` 的獨立函式。這樣你可以在不考慮撰寫和反思步驟的情況下，單獨評估搜尋結果；本範例只關注網頁搜尋步驟的輸出。

與完整工作流的實現相比，這裡有兩個關鍵差異：

* 新函式使用 **AISuite**，它會為你自動管理工具呼叫（而不是用 OpenAI SDK 手寫工具呼叫程式碼）。  
* 該函式還會告知 LLM **當前日期**，這有助於提升時間敏感查詢的相關性。  

`find_references` 的角色是從 **Arxiv**、**Tavily** 和 **Wikipedia** 等工具中**收集外部資訊**。  
由於這些結果的質量將直接影響整個工作流的輸出，因此這是可以應用**評估方法**的階段——例如，檢查返回的 URL 是否來自你的**優選域名**列表。  

In [ ]:
def find_references(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """使用外部工具（arxiv、tavily、wikipedia）執行研究任務。"""

    prompt = f"""
    你是一個研究函式，可以訪問：
    - arxiv_tool：學術論文
    - tavily_tool：通用網頁搜尋（需要時返回 JSON）
    - wikipedia_tool：百科式摘要

    任務：
    {task}

    今天是 {datetime.now().strftime('%Y-%m-%d')}。
    """.strip()

    messages = [{"role": "user", "content": prompt}]
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
    ]

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            max_turns=5,
        )
        content = response.choices[0].message.content
        return (content, messages) if return_messages else content
    except Exception as e:
        return f"[模型錯誤: {e}]"

執行下面的單元以測試研究函式。  
該任務將檢索兩篇關於黑洞科學最新進展的論文，並顯示結果。

In [ ]:
research_task = "查詢 2 篇關於黑洞科學最新進展的近期論文"
research_result = find_references(research_task)

utils.print_html(
    research_result,
    title="研究函式輸出"
)

## 4. 評估步驟 – 優選域名

並非所有網頁搜尋檢索到的來源都同樣可靠。  
在本範例中，我們聚焦於**研究工作流中的一個步驟**——`find_references` 研究步驟——並展示如何設計一個**元件級評估**，用於檢查返回的域名是否屬於預定義的**優選域名**列表。  

這是一個**具有明確按示例真實標準**的**客觀評估**案例。  
回顧評估的兩個維度：在這些維度上，我們位於**左上象限**——按示例、具有明確真實標準的客觀評估。

<img src='M4-UGL-1-Evaluations.png' width='80%'>


### 為什麼需要元件級評估？

為什麼要這樣做？  

- 如果問題出在網頁搜尋（通常是研究工作流中的**第一步**），每次重跑*整條*流水線（搜尋 → 草稿 → 反思）會非常**昂貴**且噪聲多。  
- 網頁搜尋質量的小幅提升，可能會被後續元件引入的隨機性所掩蓋。  
- 透過只評估網頁搜尋，你可以獲得該元件是否改善的**更清晰訊號**。  

當多支團隊分別負責系統的不同部分時，元件級評估也很高效：每個團隊都可以用清晰的指標最佳化自己的元件，而無需執行或等待完整的端到端測試。  

### 我們如何評估？

這裡的評估是**客觀的**，因此可以用程式碼進行評估。它具有示例特定的真實標準——本次黑洞主題的優選來源列表。要構建該評估，你將：

1. 提取 Tavily 返回的 URL。  
2. 將其與預定義的**優選域名**列表對比（例如 `arxiv.org`、`nature.com`、`nasa.gov`）。  
3. 計算**優選結果與總結果的比例**。  
4. 返回一個**透過/未透過**標誌，並附帶 Markdown 格式摘要。  

這提供了一個可復現、低成本的度量，告訴我們研究元件——也就是研究工作流中的這一步——是否從可信來源獲取資訊。



In [ ]:
# Tavily 結果的首選域名列表
TOP_DOMAINS = {
    # 通用參考 / 機構 / 出版方
    "wikipedia.org", "nature.com", "science.org", "sciencemag.org", "cell.com",
    "mit.edu", "stanford.edu", "harvard.edu", "nasa.gov", "noaa.gov", "europa.eu",

    # 電腦科學 / 人工智慧領域會議與索引
    "arxiv.org", "acm.org", "ieee.org", "neurips.cc", "icml.cc", "openreview.net",

    # 其他權威出版源
    "elifesciences.org", "pnas.org", "jmlr.org", "springer.com", "sciencedirect.com",

    # 額外域名（針對特定情況）
    "pbs.org", "nova.edu", "nvcc.edu", "cccco.edu",

    # 知名程式設計網站
    "codecademy.com", "datacamp.com"
}

def evaluate_tavily_results(TOP_DOMAINS, raw: str, min_ratio=0.4):
    """
    評估純文字研究結果是否主要來自首選域名。

    Args:
        TOP_DOMAINS (set[str]): 首選域名集合（例如 'arxiv.org', 'nature.com'）。
        raw (str): 包含 URL 的純文字或 Markdown。
        min_ratio (float): 透過評估所需的最小首選比例（例如 0.4 = 40%）。

    Returns:
        tuple[bool, str]: (flag, markdown_report)
            flag -> True 表示透過，False 表示未透過
            markdown_report -> Markdown 格式的評估摘要
    """

    # 從文字中提取 URL
    url_pattern = re.compile(r'https?://[^\s\]\)>\}]+', flags=re.IGNORECASE)
    urls = url_pattern.findall(raw)

    if not urls:
        return False, """### Evaluation — Tavily Preferred Domains
未檢測到任何 URL。
請在研究結果中包含連結。
"""

    # 統計首選域名與總數
    total = len(urls)
    preferred_count = 0
    details = []

    for url in urls:
        domain = url.split("/")[2]
        preferred = any(td in domain for td in TOP_DOMAINS)
        if preferred:
            preferred_count += 1
        details.append(f"- {url} → {'✅ 首選域名' if preferred else '❌ 非首選域名'}")

    ratio = preferred_count / total if total > 0 else 0.0
    flag = ratio >= min_ratio

    # 生成 Markdown 格式的評估報告
    report = f"""
### 評估結果 — Tavily 首選域名
- 總連結數: {total}
- 首選連結數: {preferred_count}
- 比例: {ratio:.2%}
- 閾值: {min_ratio:.0%}
- 狀態: {"✅ 透過" if flag else "❌ 未透過"}

**詳細結果：**
{chr(10).join(details)}
"""
    return flag, report


<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#dbeafe; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">  
<strong>🔎 為什麼這是客觀評估：</strong><br><br>  
從 Tavily 檢索到的每個 URL 都會與預定義的<em>優選域名</em>（<code>TOP_DOMAINS</code>）進行對比：<br>  
• 若域名匹配 → ✅ 優選<br>  
• 否則 → ❌ 非優選<br><br>  
是否透過取決於優選來源的比例是否超過設定閾值，因此會產生清晰的透過/未透過訊號。  
由於每個示例的真實標準（優選 vs 非優選）都被明確地定義，評估既<strong>客觀</strong>又<strong>可復現</strong>。  
</div>


執行該單元以顯示優選域名示例、研究結果以及評估摘要（透過/未透過及詳細資訊）。

In [ ]:
utils.print_html(json.dumps(list(TOP_DOMAINS)[:4], indent=2), title="優選域名示例")

utils.print_html("<h3>研究結果</h3>" + research_result, title="研究結果")

flag, report = evaluate_tavily_results(TOP_DOMAINS, research_result)
utils.print_html("<pre>" + report + "</pre>", title="<h3>評估摘要</h3>")

## 動手試試！

現在輪到你了。  
在本節中，你可以直接試驗**研究步驟**及其**評估**：  

* **主題**：選擇一個不同的研究主題。  
* **優選域名**：編輯或擴充套件 `TOP_DOMAINS` 列表。  
* **評估比例**：調整 `min_ratio`（例如 0.4 = 至少 40% 為優選來源）。  

修改後重新執行下面的單元，觀察評估如何變化。


In [ ]:
# === 5.1. 自主嘗試：主題、比例與優選域名 ===
# 在執行單元前編輯這些引數

topic = "黑洞科學的最新進展"   # <- 在此修改主題
min_ratio = 0.4                   # <- 修改閾值（0.0–1.0）
run_reflection = True             # <- 設為 False 跳過步驟 4

# 簡短的優選域名列表（按需編輯或擴充套件）
TOP_DOMAINS = {
    "wikipedia.org", "nature.com", "science.org", "arxiv.org",
    "nasa.gov", "mit.edu", "stanford.edu", "harvard.edu"
}

# 顯示優選域名示例
import json
utils.print_html(
    json.dumps(sorted(list(TOP_DOMAINS)), indent=2),
    title="<h3>優選域名示例</h3>"
)

# 1) 研究
research_task = f"查詢 {topic} 的 2–3 篇關鍵論文與可靠綜述。"
research_output = find_references(research_task)
utils.print_html(research_output, title=f"<h3>{topic} 的研究結果</h3>")

# 2) 評估來源（優選域名比例）
flag, eval_md = evaluate_tavily_results(TOP_DOMAINS, research_output, min_ratio=min_ratio)
utils.print_html("<pre>" + eval_md + "</pre>", title="<h3>評估摘要</h3>")

## 5. 要點總結

* 你剛剛學習瞭如何評估**一個元件**的效能：即 `find_references` 研究步驟。  
* 你的元件級評估檢查了檢索到的 URL 是否在預定義的**優選域名**列表中。  
* 這是一個具有明確**按示例真實標準**的**客觀評估**示例。  
* 為構建評估集，你可以設計約 10 個涵蓋不同主題（天文學、機器人、金融等）的提示，並為每個主題定義優選域名。  
* 與優選域名列表匹配的來源百分比提供了有用的**指標**，可指導改進，例如調整提示或工具引數。  
* 這種方法比評估包含反思與重寫的整篇文章**更簡單也更便宜**，因為你僅聚焦於網頁搜尋元件。  

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 **恭喜！**  

你設計了一個**元件級評估**，使你的研究代理更加可靠。  
透過直接檢查來源質量，你引入了一個**客觀、可復現、且高價效比**的保障。  

這正是元件級評估的核心理念：*元件級評估*讓你無需評估整條流水線就能測試 AI 系統的單個部分。  

</div>


